In [3]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
from tqdm import tqdm

In [4]:
# Load environment variables
load_dotenv()
API_KEY = os.getenv('XC_API_KEY')

def fetch_xc_page(query, page=1, per_page=500):
    """
    Xeno-canto API v3 requires 'query' and 'key' as URL parameters.
    """
    if not API_KEY:
        raise ValueError("API Key not found. Check your .env file.")

    base_url = "https://xeno-canto.org/api/3/recordings"
    
    # API v3 uses 'key' as a query parameter, not a header
    params = {
        'query': query,
        'key': API_KEY,
        'page': page,
        'per_page': per_page
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=15)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error on page {page}: {e}")
        return None

def get_northern_data():
    # v3 is stricter: tags only. 
    # Use 'cnt:canada' as the primary tag.
    query = 'cnt:canada'
    all_recs = []
    
    print(f"Requesting API v3 data for: {query}")
    # Requesting 500 per page to minimize API calls
    data = fetch_xc_page(query, page=1, per_page=500)

    if not data or 'recordings' not in data:
        print(f"API Error or no recordings. Full Response: {data}")
        return pd.DataFrame()

    total_recordings = data.get('numRecordings', 0)
    total_pages = int(data.get('numPages', 1))
    print(f"Found {total_recordings} recordings. Fetching {total_pages} pages...")
    
    all_recs.extend(data['recordings'])
    
    # Loop through remaining pages
    # Note: Keep the range small (e.g., 5) for your first test run
    for p in range(2, total_pages + 1):
        print(f"Fetching page {p}/{total_pages}...")
        p_data = fetch_xc_page(query, page=p, per_page=500)
        if p_data and 'recordings' in p_data:
            all_recs.extend(p_data['recordings'])
        else:
            break
            
    return pd.DataFrame(all_recs)

def clean_to_unified_schema(df):
    if df.empty:
        return pd.DataFrame()

    # XC Quality mapping (A-E to 5-1)
    q_map = {'A': 5, 'B': 4, 'C': 3, 'D': 2, 'E': 1, 'no score': 0}
    
    # Map the v3 response to your required deliverable format
    unified_df = pd.DataFrame()
    
    # Ensure ID exists for the filepath
    ids = df['id'].astype(str)
    
    unified_df['filepath'] = "audio/xc/" + ids + "." + df.get('format', 'mp3')
    unified_df['species_code'] = df.get('en', 'unknown').str.lower().str.replace(' ', '_')
    unified_df['common_name'] = df.get('en', 'Unknown')
    unified_df['vocalization_type'] = df.get('type', 'unknown')
    unified_df['quality_rating'] = df.get('q', 'no score').map(q_map).fillna(0)
    unified_df['duration'] = df.get('length', '0:00')
    unified_df['source'] = 'xeno-canto'
    
    return unified_df

# --- EXECUTION ---
raw_df = get_northern_data()

if not raw_df.empty:
    final_csv = clean_to_unified_schema(raw_df)
    final_csv.to_csv("xc_metadata_unified.csv", index=False)
    print(f"Success! {len(final_csv)} rows saved to xc_metadata_unified.csv")
else:
    print("No data retrieved. Verify your API key and query tag.")

Requesting API v3 data for: cnt:canada
Found 17787 recordings. Fetching 36 pages...
Fetching page 2/36...
Fetching page 3/36...
Fetching page 4/36...
Fetching page 5/36...
Fetching page 6/36...
Fetching page 7/36...
Fetching page 8/36...
Fetching page 9/36...
Fetching page 10/36...
Fetching page 11/36...
Fetching page 12/36...
Fetching page 13/36...
Fetching page 14/36...
Fetching page 15/36...
Fetching page 16/36...
Fetching page 17/36...
Fetching page 18/36...
Fetching page 19/36...
Fetching page 20/36...
Fetching page 21/36...
Fetching page 22/36...
Fetching page 23/36...
Fetching page 24/36...
Fetching page 25/36...
Fetching page 26/36...
Fetching page 27/36...
Fetching page 28/36...
Fetching page 29/36...
Fetching page 30/36...
Fetching page 31/36...
Fetching page 32/36...
Fetching page 33/36...
Fetching page 34/36...
Fetching page 35/36...
Fetching page 36/36...
Success! 17787 rows saved to xc_metadata_unified.csv
